[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/altair-certified/notebooks/day-05-transforms.ipynb#scrollTo=1a2b3c4d)

---
# Day 5 · Transforms
**certified-journeys / altair-certified** · Day 5 · Transforms

> **Goal for today:** Use Vega-Lite transforms — filter, calculate, aggregate, bin, window, and fold — to reshape data entirely inside the chart spec, without modifying your pandas DataFrame.

In [ ]:
%pip install -q altair vega-datasets

## Step 1 · transform_filter — row-level filtering

`transform_filter` removes rows from the data **before any mark or aggregation is applied**. You can pass:

- An `alt.datum` expression string: `alt.datum.field > value`
- A Vega expression string: `'datum.field > value'`
- A selection object (for interactive filtering)

`alt.datum` is a special object that references field values in the current data row — think of it as the row variable in a filter predicate.

In [ ]:
import altair as alt
from vega_datasets import data
import pandas as pd
import math

cars = data.cars()

# Filter to only show cars with Horsepower > 100 using alt.datum
alt.Chart(cars).mark_point().encode(
    x='Horsepower:Q',
    y='Miles_per_Gallon:Q',
    color='Origin:N'
).transform_filter(
    alt.datum.Horsepower > 100   # row-level predicate — no pandas needed
).properties(title='Cars with Horsepower > 100', width=380)

### What just happened?

- `alt.datum.Horsepower > 100` creates a Vega expression predicate — it runs in the browser, not in Python.
- The original `cars` DataFrame is **unchanged** — filtering is purely declarative in the spec.
- **When to use vs pandas:** prefer `transform_filter` when data comes from a URL or when you want the filter to be part of the reusable chart spec.

## Step 2 · transform_calculate — derived fields

`transform_calculate` adds a new computed column to the data stream using a Vega expression. This is the Vega-Lite equivalent of `df['new_col'] = ...`.

Common patterns:

| Operation | Vega expression |
|-----------|-----------------|
| Log transform | `'log(datum.value)'` |
| Ratio | `'datum.a / datum.b'` |
| String concat | `'datum.first + " " + datum.last'` |
| Conditional | `'datum.x > 0 ? "positive" : "negative"'` |

In [ ]:
# Add a log-transformed horsepower field via transform_calculate
alt.Chart(cars).mark_point(opacity=0.7).encode(
    x=alt.X('log_hp:Q', title='log(Horsepower)'),   # new field from calculate
    y='Miles_per_Gallon:Q',
    color='Origin:N'
).transform_calculate(
    log_hp='log(datum.Horsepower)'   # Vega's log() is natural log (base e)
).transform_filter(
    alt.datum.Horsepower > 0         # guard against log(0)
).properties(title='MPG vs log(Horsepower)', width=380)

### What just happened?

- `transform_calculate(log_hp='log(datum.Horsepower)')` creates a new virtual column called `log_hp`.
- We chain `transform_filter` **after** `transform_calculate` to guard against `log(0)` — transforms apply in the order they're chained.
- **The new field only exists in the Vega-Lite data flow**, not in the `cars` DataFrame — the source data is immutable.

## Step 3 · transform_aggregate — group-level summaries

`transform_aggregate` is the Vega-Lite equivalent of pandas `groupby().agg()`. It collapses the data to one row per group, computing summary statistics.

Key differences from encoding-level aggregation (`'mean(field):Q'`):

- `transform_aggregate` **creates new named fields** you can reference in any encoding.
- It collapses the row count — useful when downstream transforms need the aggregated values.
- You can compute **multiple aggregations** in one transform.

In [ ]:
# Compute mean Horsepower AND count of cars per Origin in one transform
alt.Chart(cars).mark_bar().encode(
    x=alt.X('Origin:N', title='Country'),
    y=alt.Y('mean_hp:Q', title='Mean Horsepower'),
    tooltip=[
        alt.Tooltip('Origin:N'),
        alt.Tooltip('mean_hp:Q', format='.1f', title='Avg HP'),
        alt.Tooltip('car_count:Q', title='# Cars')
    ]
).transform_aggregate(
    mean_hp='mean(Horsepower)',   # aggregation op(field) syntax
    car_count='count()',
    groupby=['Origin']            # group dimension(s)
).properties(title='Mean HP and Count per Origin', width=300)

### What just happened?

- `transform_aggregate` collapses the 406-row `cars` data to 3 rows — one per Origin.
- `mean_hp` and `car_count` are new field names you defined; they're now usable in any encoding or tooltip.
- **`count()` requires no field name** — it counts rows in each group.

## Step 4 · transform_bin — in-spec histograms

`transform_bin` bins a quantitative field into equal-width buckets, creating a `field_binned` and `field_binned_end` column pair. This lets you build a histogram **without pre-binning in pandas**.

```python
transform_bin('hp_bin', 'Horsepower', bin=alt.Bin(maxbins=20))
```

The `maxbins` hint tells Vega-Lite the maximum number of bins to generate — the actual bin width is chosen to produce round numbers.

In [ ]:
# Build a histogram of Horsepower entirely with transforms (no pandas binning)
alt.Chart(cars).mark_bar().encode(
    x=alt.X('hp_bin:Q',
            bin='binned',           # tell Altair the data is already binned
            title='Horsepower'),
    x2='hp_bin_end:Q',              # right edge of each bin
    y=alt.Y('count:Q', title='Number of Cars')
).transform_bin(
    as_=['hp_bin', 'hp_bin_end'],   # name for start and end fields
    field='Horsepower',
    bin=alt.Bin(maxbins=15)
).transform_aggregate(
    count='count()',
    groupby=['hp_bin', 'hp_bin_end']
).properties(title='Horsepower Distribution (binned in Vega-Lite)', width=400)

### What just happened?

- `transform_bin` creates two fields (`hp_bin`, `hp_bin_end`) representing bin boundaries.
- We then `transform_aggregate` to count rows per bin — this is the full histogram pipeline in Vega-Lite.
- `bin='binned'` on the X encoding tells Altair the field is already pre-binned, so it renders bar widths correctly using `x2`.
- **The key benefit:** this approach works when data comes from a URL — no Python needed to compute the histogram.

## Step 5 · transform_window — rolling averages

`transform_window` computes window functions (running sum, rolling mean, rank, etc.) over a sorted data stream. You specify:

- `window`: list of `{op, field, as}` objects
- `frame`: `[start, end]` offset relative to the current row — `[-6, 0]` = previous 6 rows through current
- `sort`: field to sort the window over

In [ ]:
# Compute a 7-day rolling average of precipitation using transform_window
seattle = data.seattle_weather()

base = alt.Chart(seattle).encode(
    x=alt.X('date:T', title='Date')
)

raw_line = base.mark_line(opacity=0.25, color='steelblue').encode(
    y=alt.Y('precipitation:Q', title='Precipitation (mm)')
)

rolling_line = base.mark_line(color='firebrick', strokeWidth=2).encode(
    y='rolling_precip:Q'
).transform_window(
    rolling_precip='mean(precipitation)',   # compute rolling mean
    frame=[-6, 0],                           # 7-day window: 6 prior rows + current
    sort=[{'field': 'date'}]                 # ensure chronological order
)

(raw_line + rolling_line).properties(
    title='Seattle Precipitation: Raw vs 7-day Rolling Average',
    width=500
)

### What just happened?

- `transform_window(rolling_precip='mean(precipitation)', frame=[-6, 0])` computes a 7-day rolling mean.
- `frame=[-6, 0]` means: include 6 rows before the current row plus the current row (7 total).
- The `sort` parameter is **critical** — without it, the window operates over an arbitrary row order.
- We overlay raw and rolling lines using the `+` operator (layering — covered in Day 6).

## Step 6 · transform_fold — wide to long inside Vega-Lite

`transform_fold` pivots multiple wide columns into a single `key`/`value` long-form pair — without modifying your DataFrame. This is the Vega-Lite equivalent of `pd.melt()`.

```python
transform_fold(['col_a', 'col_b', 'col_c'], as_=['metric', 'value'])
```

After folding, you can use `metric:N` as a colour channel to plot multiple series on the same axes.

In [ ]:
# Plot temp_max and temp_min on the same chart using transform_fold
# Seattle weather has both columns — fold them into a single 'metric'/'temp' pair

alt.Chart(seattle).mark_line().encode(
    x=alt.X('date:T', title='Date'),
    y=alt.Y('temp:Q', title='Temperature (°C)'),
    color=alt.Color('metric:N', title='Metric')  # 'metric' = key column from fold
).transform_fold(
    ['temp_max', 'temp_min'],   # wide columns to fold
    as_=['metric', 'temp']      # output key/value column names
).properties(title='Seattle Daily Temp Range (folded wide→long)', width=500)

### What just happened?

- `transform_fold` doubled the row count — each original row now has two rows: one for `temp_max`, one for `temp_min`.
- The new `metric` column contains `'temp_max'` or `'temp_min'` as strings, which Altair uses for the color legend.
- **The source DataFrame is unmodified** — fold happens in the Vega-Lite spec, not Python.
- This is especially powerful with URL-sourced data or when you want to keep transformations declarative.

In [ ]:
# Challenge: Chaining transforms
#
# Using the 'cars' dataset:
#   1. Use transform_calculate to add a 'weight_kg' field: Weight_in_lbs * 0.453592
#   2. Use transform_filter to keep only cars from 'USA' (alt.datum.Origin == 'USA')
#   3. Use transform_bin to bin 'weight_kg' into at most 10 bins
#   4. Use transform_aggregate to count rows per bin
#   5. Render as a bar chart (histogram of weight_kg for US cars)
#
# Hint: chain transforms in order — calculate → filter → bin → aggregate

# Your solution here:
# alt.Chart(cars).mark_bar().encode(
#     x=alt.X('wkg_bin:Q', bin='binned', title='Weight (kg)'),
#     x2='wkg_bin_end:Q',
#     y=alt.Y('count:Q', title='Count')
# ).transform_calculate(
#     weight_kg=___
# ).transform_filter(
#     ___
# ).transform_bin(
#     as_=['wkg_bin', 'wkg_bin_end'],
#     field=___,
#     bin=alt.Bin(maxbins=___)
# ).transform_aggregate(
#     count='count()',
#     groupby=___
# ).properties(title='US Car Weights (kg)', width=380)

---
## Day 5 key concepts recap

| Transform | Equivalent pandas op | Key parameter |
|-----------|---------------------|---------------|
| `transform_filter` | `df[mask]` | `alt.datum.field op value` |
| `transform_calculate` | `df['col'] = expr` | `new_field='vega_expression'` |
| `transform_aggregate` | `groupby().agg()` | `groupby=['field']` |
| `transform_bin` | `pd.cut()` | `bin=alt.Bin(maxbins=N)` |
| `transform_window` | `rolling().mean()` | `frame=[start, end]` |
| `transform_fold` | `pd.melt()` | `as_=['key', 'value']` |

> **Tip:** Prefer Vega-Lite transforms when your data comes from a URL — they run in the browser without Python. Use pandas transforms when you need complex logic or when data is already in-memory.

---
## What's next
**Day 6** → Compound Charts — layering with `+`, concatenation with `|` and `&`, facets, and repeat/SPLOM.

Mark Day 5 complete in your [tracker](../index.html).